# Filtrado colaborativo: _Bernoulli Matrix Factorization_

La Factorización Matricial tradicional (SVD, NMF) aborda la recomendación como un problema de regresión. Es decir, el producto escalar de los factores latentes intenta predecir un valor continuo (por ejemplo, "el usuario le dará un 3.8").

El problema de la regresión es que no te dice cuán seguro está el modelo. Si predice un 4, no sabes si es un 4 rotundo apoyado por cientos de datos, o una estimación ruidosa basada en casi nada.

BeMF transforma la matriz en un problema de clasificación probabilística:

- Coge el rango discreto de notas (por ejemplo, 1, 2, 3, 4 y 5 estrellas) y las convierte usando One-Hot Encoding.
- En lugar de hacer una factorización para adivinar el número, entrena factorizaciones asumiendo que cada voto sigue una distribución de Bernoulli (es un éxito o un fracaso para esa nota específica).
- Pasa el producto escalar de los vectores latentes por una función de activación (tipo sigmoide/logística) para obtener una probabilidad entre 0 y 1.

Al tratar cada nota como una clase, la salida de BeMF para un usuario y una película no es un número pelado, sino una distribución de probabilidades.

Por ejemplo, el modelo podría devolver esto para una película concreta:

- 1 estrella: 2%
- 2 estrellas: 5%
- 3 estrellas: 10%
- 4 estrellas: 80%
- 5 estrellas: 3%

La predicción final es un 4 (la clase con mayor probabilidad), pero lo interesante reside en ese 80%. Ese porcentaje es tu Métrica de Fiabilidad. En un entorno de producción real, puedes configurar el sistema para que solo recomiende ítems donde la fiabilidad supere el 75%. Si las probabilidades están muy planas (ej. 20% para cada estrella), el modelo admite que "no tiene ni idea", lo que evita hacer recomendaciones arriesgadas que frustren al usuario.

## Entrenamiento del modelo

A diferencia de los modelos tradicionales (PMF, NMF) que tratan la recomendación como un problema de regresión para adivinar un número exacto, **BeMF transforma la recomendación en un problema de clasificación probabilística múltiple**. 

El objetivo de BeMF es calcular la probabilidad independiente de que un usuario asigne cada una de las posibles puntuaciones a un ítem, asumiendo que cada voto sigue una distribución de Bernoulli (éxito/fracaso).

Dado un conjunto $S$ de votos posibles, la matriz original de valoraciones $R$ se divide en $|S|$ matrices binarias independientes:

$$R_{u,i}^{(s)} = \begin{cases} 1 & \text{si } R_{u,i}=s \\ 0 & \text{en caso contrario} \end{cases}$$

para $s \in S$.

Para cada posible nota $s\in S$, el modelo aprende un conjunto independiente de factores latentes para los usuarios ($U^{(s)}$) y para los ítems ($V^{(s)}$).

La probabilidad de que el usuario $u$ asigne la nota $s$ al ítem $i$ se modela aplicando una función ($\psi: \mathbb{R} \rightarrow [0, 1]$) al producto escalar de sus vectores latentes:

$$\hat{p}_{u,i}^{(s)} = P(r_{u,i} = s) = \psi( (U_u^{(s)})^T V_i^{(s)} )$$

Luego

$$
p(R_{u,i}^s | U_u^s, V_i^s) = \begin{cases} \psi(U_u^s \cdot V_i^s) & \text{si } R_{u,i}^s = 1 \\ 1 - \psi(U_u^s \cdot V_i^s) & \text{si } R_{u,i}^s = 0 \end{cases}
$$

Dada una muestra concreta $R_{u,i}^s$, obtenemos la verosimilitud:

$$ \mathfrak{L}(R_{u,i}^s | U^s, V^s) = \prod_{R_{u,i}^s \neq \bullet} p(R_{u,i}^s | U_u^s, V_i^s) = \left ( \prod_{R_{u,i}^s = 1} \psi(U_u V_i) \right) \left ( \prod_{R_{u,i}^s = 0} 1 - \psi(U_u V_i) \right) $$

Aplicando logaritmo, el log-likelihood $l$ queda:

$$ l(R_{u,i}^s | U^s, V^s) = \sum_{R_{u,i}^s = 1} \psi(U_u V_i) + \prod_{R_{u,i}^s = 0} 1 - \psi(U_u V_i) $$

Asumimos _priors_ normales esféricas con media $0$ y $\sigma_u, \sigma_v$ como hiperparámetros:

$$
\begin{split}
p(U_u^s) &= \frac{1}{\sigma_u \sqrt{2\pi}} \mathit{e}^{-\frac{\left\| U_u \right\|^2}{2\sigma_u^2}}\\
p(V_i^s) &= \frac{1}{\sigma_v \sqrt{2\pi}} \mathit{e}^{-\frac{\left\| V_i \right\|^2}{2\sigma_v^2}}
\end{split}
$$

La verosimilitud de estos _priors_ es:

$$
\begin{split}
\mathfrak{L}(U^s) &= \prod_{u=1}^N p(U_u^s) = \frac{1}{\sigma_u^N (2\pi)^{N/2}} \mathit{e}^{-\frac{\sum_{u=1}^N \left\| U_u^s \right\|^2}{2\sigma_u^2}}\\
\mathfrak{L}(V^s) &= \prod_{i=1}^M p(V_i^s) = \frac{1}{\sigma_v^N (2\pi)^{N/2}} \mathit{e}^{-\frac{\sum_{i=1}^M \left\| V_i^s \right\|^2}{2\sigma_v^2}}
\end{split}
$$

para $N, M$ siendo el número de usuarios e items, respectivamente. Aplicando logaritmo obtenemos:

$$
\begin{split}
l(U^s) &= - \frac{1}{2\sigma_u^2} \sum_{u=1}^N \left\| U_u^s \right\|^2 + C_u\\
l(V^s) &= - \frac{1}{2\sigma_v^2} \sum_{i=1}^M \left\| V_i^s \right\|^2 + C_v
\end{split}
$$

donde $C_u=-N \log (\sigma_u \sqrt{2\pi})$ y $C_v=-N \log (\sigma_v \sqrt{2\pi})$ y constantes.

Una vez definidos los _priors_ podemos definir la verosimilitud posterior:

$$ \mathfrak{L}(R^s) = \mathfrak{L}(R^s|U^s, V^s) \mathfrak{L}(U^s) \mathfrak{L}(V^s)$$

y, aplicando logaritmos, llegar a la _log-likelihood_ posterior:

$$ l(R^s) = l(R^s|U^s, V^s) + l(U^s) + l(V^s) = \sum_{R_{u,i}=1} \log (\psi (U_u V_i)) + \sum_{R_{u,i}=0} \log (1 - \psi (U_u V_i)) - \frac{1}{2\sigma_u^2} \sum_{u=1}^N \left\| U_u^s \right\|^2 - \frac{1}{2\sigma_v^2} \sum_{i=1}^M \left\| V_i^s \right\|^2 + C$$

donde $C = C_u + C_v$ es una constante.

Para una votación $s \in S$ particular fijamos $R=R^s$. El estimador de máxima verosimilitud se obtiene maximizando el _log-likelihood_ posterior $l(R)$. Para convertirlo a un problema de minimización, fijamos $\eta_U=\frac{1}{\sigma_u^2}$ y $\eta_V=\frac{1}{\sigma_v^2}$ y multiplicamos por $-1$, quedando la función de coste como:

$$
\mathfrak{F}(U,V) = - \sum_{R_{u,i}=1} \log (\psi (U_u V_i)) - \sum_{R_{u,i}=0} \log (1 - \psi (U_u V_i)) + \frac{\eta_U}{2}\sum_{u=1}^N \left\| U_u^s \right\|^2 + \frac{\eta_V}{2}\sum_{i=1}^M \left\| V_i^s \right\|^2
$$

Calculando las derivadas parciales:

$$
    \frac{\partial  \mathcal{F}}{\partial U_{u_0,a}} = -\sum_{\left\{i \,|\, R_{u_0,i} = 1\right\}} \frac{\psi'(U_{u_0}V_i)}{\psi(U_{u_0}V_i)}V_{i,a} + \sum_{\left\{i \,|\, R_{u_0,i} = 0\right\}} \frac{\psi(U_{u_0}V_i)}{1-\psi(U_{u_0}V_i)}V_{i,a} + \eta_U U_{u_0,a},
$$
$$
    \frac{\partial  \mathcal{F}}{\partial V_{i_0,b}} = -\sum_{\left\{u \,|\, R_{u,i_0} = 1\right\}} \frac{\psi'(U_{u}V_{i_0})}{\psi(U_{u}V_{i_0})}U_{u,b} + \sum_{\left\{u \,|\, R_{u,i_0} = 0\right\}} \frac{\psi'(U_{u}V_{i_0})}{1-\psi(U_{u}V_{i_0})}U_{u,b} + \eta_V V_{i_0,b}.
$$

Por simplicidad, consideramos $\eta_U = \eta_V = \eta$ y $\psi(x) = \mathit{logit}(x) = \frac{1}{1-\mathit{e}^x}$, con $ \mathit{logit}'(x) = \mathit{logit}(x)(1- \mathit{logit}) $. Luego, las ecuaciones de actualización quedan como:

$$
\begin{split}
U_u^{T+1} &= U_u^{T} + \gamma \left(\sum_{\left\{i \,|\, R_{u,i} = 1\right\}} \frac{\mathrm{logit}(U_{u}V_i)(1-\mathrm{logit}(U_{u}V_i))}{\mathrm{logit}(U_{u}V_i)}V_{i} - \sum_{\left\{i \,|\, R_{u,i} = 0\right\}} \frac{\mathrm{logit}(U_{u}V_i)(1-\mathrm{logit}(U_{u}V_i))}{1-\mathrm{logit}(U_{u}V_i)}V_{i} - \eta U_{u}\right)\\
V_i^{T+1} &= V_i^{T} + \gamma \left(\sum_{\left\{u \,|\, R_{u,i} = 1\right\}} \frac{\mathrm{logit}(U_{u}V_i)(1-\mathrm{logit}(U_{u}V_i))}{\mathrm{logit}(U_{u}V_{i})}U_{u} - \sum_{\left\{u \,|\, R_{u,i} = 0\right\}} \frac{\mathrm{logit}(U_{u}V_i)(1-\mathrm{logit}(U_{u}V_i))}{1-\mathrm{logit}(U_{u}V_{i})}U_{u} - \eta V_{i}\right)
\end{split}
$$

Que simplificando resultan en:

$$
\begin{split}
U_u^{T+1} &= U_u^{T} + \gamma \left(\sum_{\left\{i \,|\, R_{u,i} = 1\right\}} (1-\mathrm{logit}(U_{u}V_i))V_{i} - \sum_{\left\{i \,|\, R_{u,i} = 0\right\}} \mathrm{logit}(U_{u}V_i)V_{i} - \eta U_{u}\right)\\
V_i^{T+1} &= V_i^{T} + \gamma \left(\sum_{\left\{u \,|\, R_{u,i} = 1\right\}} (1-\mathrm{logit}(U_{u}V_i))U_{u} - \sum_{\left\{u \,|\, R_{u,i} = 0\right\}} \mathrm{logit}(U_{u}V_i)U_{u} - \eta V_{i}\right)
\end{split}
$$

## Carga del dataset

Para ilustrar mejor el funcionamiento el algoritmo BNMF, vamos a desarrollar una implementación del mismo.

Para ello usaremos el dataset de [MovieLens 100K](https://grouplens.org/datasets/movielens/) que contiene 100.000 votos de 943 usuarios sobre 1682 películas. Este dataset ha sido dividido en votaciones de entrenamiento (80%) y votaciones de test (20%). Además, los códigos de usuarios e items, han sido modificados para que comience en 0 y terminen en el número de (usuarios / items) - 1.


In [ ]:
import urllib
import random
import numpy as np
from math import exp

In [ ]:
NUM_USERS = 943
NUM_ITEMS = 1682

MIN_RATING = 1
MAX_RATING = 5

Y cargamos el dataset:

In [ ]:
ratings = [[None for _ in range(NUM_ITEMS)] for _ in range(NUM_USERS)] 

training_file = urllib.request.urlopen(r"https://raw.githubusercontent.com/KNODIS-Research-Group/ml-notebooks/refs/heads/main/sistemas%20de%20recomendacion/datasets/movielens-100k.training")
for line in training_file:
  [u, i, rating] = line.decode("utf-8").split("::")
  ratings[int(u)][int(i)] = int(rating)

Del mismo modo, cargamos la matriz de votaciones de test:

In [ ]:
test_ratings = [[None for _ in range(NUM_ITEMS)] for _ in range(NUM_USERS)] 

test_file = urllib.request.urlopen(r"https://raw.githubusercontent.com/KNODIS-Research-Group/ml-notebooks/refs/heads/main/sistemas%20de%20recomendacion/datasets/movielens-100k.test")
for line in test_file:
  [u, i, rating] = line.decode("utf-8").split("::")
  test_ratings[int(u)][int(i)] = int(rating)

## Inicialización del modelo

Definimos los parámetros necesarios para implementar la factorización matricial mediante BeMF.

In [ ]:
NUM_FACTORS = 7
LEARNING_RATE = 0.001
REGULARIZATION = 0.1

Inicializamos las matrices de factores con valores uniformes aleatorios en el intervalo \[0, 1].

In [ ]:
num_ratings = MAX_RATING - MIN_RATING + 1
p = np.random.uniform(0, 1, (num_ratings, NUM_USERS, NUM_FACTORS))
q = np.random.uniform(0, 1, (num_ratings, NUM_ITEMS, NUM_FACTORS))

## Cálculo de las predicciones

Como hemos comentado, calcular la predicción del voto del usuario *u* al item *i* implicar realizar el producto escalar de sus vectores de factores latentes y la pasamos por la función logística. Las siguientes funciones realizan esta operación:


In [ ]:
def logit(x):
  # Estabilizar la función logística para evitar overflows
  if x < -100: return 0
  if x > 100: return 1
  return 1 / (1 + exp(-x))

def compute_probability (p, q, u, i, r):
  # p: (num_ratings, NUM_USERS, NUM_FACTORS)
  # q: (num_ratings, NUM_ITEMS, NUM_FACTORS)
  num_ratings = MAX_RATING - MIN_RATING + 1
  logits = []
  for k in range(num_ratings):
    logits.append(logit(np.dot(p[k][u], q[k][i])))
  
  suma = sum(logits)
  if suma == 0: return 1 / num_ratings # Evitar división por cero
  return logits[r - MIN_RATING] / suma

def compute_prediction (p, q, u, i):
  max_prob = -1
  best_r = MIN_RATING
  for r in range(MIN_RATING, MAX_RATING + 1):
    prob = compute_probability(p, q, u, i, r)
    if prob > max_prob:
      max_prob = prob
      best_r = r
  return best_r

## Aprendizaje de los factores latentes

El proceso de entrenamiento implicar aplicar las operaciones de actualización de las matrices de factores hasta que el algoritmo converja. En general, esta convergencia suele prefijarse como el número de iteraciones que realizamos sobre las operaciones de actualización:

In [ ]:
NUM_ITERATIONS = 20
num_ratings = MAX_RATING - MIN_RATING + 1

for it in range(NUM_ITERATIONS):
  print("Iteración " + str(it + 1) + " de " + str(NUM_ITERATIONS))
    
  # Iteramos sobre los usuarios e ítems que tienen votación
  for u in range(NUM_USERS):
    for i in range(NUM_ITEMS):
      rating_val = ratings[u][i]
      if rating_val is not None:
        # Para cada posible nota s, actualizamos p[s][u] y q[s][i]
        for s_idx in range(num_ratings):
          s_val = s_idx + MIN_RATING
          
          dot_val = np.dot(p[s_idx][u], q[s_idx][i])
          l = logit(dot_val)
          
          if rating_val == s_val:
            # R_{u,i} = 1: gradiente es (1 - logit)*V - eta*U
            p_grad = (1 - l) * q[s_idx][i] - REGULARIZATION * p[s_idx][u]
            q_grad = (1 - l) * p[s_idx][u] - REGULARIZATION * q[s_idx][i]
          else:
            # R_{u,i} = 0: gradiente es (-logit)*V - eta*U
            p_grad = -l * q[s_idx][i] - REGULARIZATION * p[s_idx][u]
            q_grad = -l * p[s_idx][u] - REGULARIZATION * q[s_idx][i]
          
          p[s_idx][u] += LEARNING_RATE * p_grad
          q[s_idx][i] += LEARNING_RATE * q_grad


In [ ]:
predictions = [[None for _ in range(NUM_ITEMS)] for _ in range(NUM_USERS)] 

# Rellenamos la matriz de predicciones
for u in range(NUM_USERS):
  for i in range(NUM_ITEMS):
    if test_ratings[u][i] is not None:
      predictions[u][i] = compute_prediction(p, q, u, i)

Y, a continuación, calculamos el MAE:

In [ ]:
def get_user_mae (u):
  mae = 0
  count = 0
  
  for i in range(NUM_ITEMS):
    if test_ratings[u][i] != None and predictions[u][i] != None:
      mae += abs(test_ratings[u][i] - predictions[u][i])
      count += 1
  
  if count > 0:
    return mae / count
  else:
    return None

In [ ]:
def get_mae ():
  mae = 0
  count = 0
  
  for u in range(NUM_USERS):
    user_mae = get_user_mae(u)
      
    if user_mae != None:
      mae += user_mae
      count += 1
  
  
  if count > 0:
    return mae / count
  else:
    return None   

In [ ]:
mae = get_mae()
print("System MAE = " + str(mae))

## Referencias

Ortega, F., Lara-Cabrera, R., González-Prieto, Á., & Bobadilla, J. (2021). **Providing reliability in recommender systems through Bernoulli Matrix Factorization**. *Information Sciences*, 553, 110-128. doi: [10.1016/j.ins.2020.12.001](https://dx.doi.org/10.1016/j.ins.2020.12.001)

---

*Este documento ha sido desarrollado por **Raúl Lara-Cabrera**. Dpto. Sistemas Informáticos, ETSI de Sistemas Informáticos, Universidad Politécnica de Madrid.*

*Última actualización: Marzo de 2026*


<p xmlns:cc="http://creativecommons.org/ns#" >This work is licensed under <a href="http://creativecommons.org/licenses/by-nc/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1"></a></p>